In [ ]:

packages = c("reticulate", "tidyverse", "stringr", "kableExtra")

for (pkg in packages) {
    library(pkg, character.only = TRUE, warn.conflicts = FALSE, quietly = TRUE, verbose = FALSE)
}
options(dplyr.width = Inf, dplyr.print_max = 1e9)
options(stringsAsFactors = FALSE)

#### script parameters ####

EMPTY_TEX_STRING = "---"

# ex_df = sudoku_df[
#     ((sudoku_df['concept_noise'] == 0.15) | (sudoku_df['concept_noise'] == 0.0)) 
#     & (sudoku_df['split'] == 'test') 
#     & ((sudoku_df['concept_missing'] == 0.3) | (sudoku_df['concept_missing'] == 0.0))
#     & (sudoku_df['tau'] == 0.05)
#     ]
# ex_df.loc[:, 'has_concept_noise'] = ex_df['concept_noise'] > 0.0


In [ ]:
sudoku_results <- "../../../results/big_demo/conceptual_safeguards_sudoku.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.05
SPLIT = "test"

sudoku_df <- read_csv(sudoku_results, show_col_types = FALSE)
sudoku_df <- sudoku_df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0)

robot_cs_results <- "../../../results/big_demo/conceptual_safeguards_robot_updated.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

robot_df <- read_csv(robot_cs_results, show_col_types = FALSE)
robot_df <- robot_df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == round(1.05 - target_accuracy_value, 2))) %>%
    mutate(has_concept_noise = concept_noise > 0.0)

In [ ]:
data_order = c("sudoku", "robot")
missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")
difficulty_order = c("easy", "medium", "hard")

df <- rbind(sudoku_df, robot_df)
df$data_name <- factor(df$data_name, levels = data_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$metric <- factor(df$metric, levels = metric_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)
df <- df %>%
    select(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df = table_stats_df %>%
    arrange(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = c(metric, c(has_concept_noise)),
        values_from = svalue,
        names_vary = "slowest"
    )


In [ ]:
cells_df

In [ ]:
unite_and_format <- function(data, prefix, new_col_name) {
  data %>%
    unite(!!sym(new_col_name),
      sep = "\\\\",
      c(paste0(prefix, "_FALSE"), paste0(prefix, "_TRUE"))
    ) %>%
    mutate(!!sym(new_col_name) := sprintf("\\cell{r}{%s}", !!sym(new_col_name)))
}
table_df <- cells_df %>%
  group_by(data_name, target_accuracy_label, concept_missing_mech) %>%
  unite_and_format("coverage_before", "cov_before") %>%
  unite_and_format("coverage_after", "cov_after") %>%
  unite_and_format("selective_acc_before", "acc_before") %>%
  unite_and_format("selective_acc_after", "acc_after") %>%
  ungroup()

In [ ]:
table_df

In [ ]:
kable_df

In [ ]:
kable_df <- table_df %>%
    mutate(
        concept_noise = "\\conceptNoise{}"
    ) %>%
    pivot_wider(
        names_from = c("concept_missing_mech"),
        values_from = c("cov_before", "cov_after", "acc_before", "acc_after"),
        names_vary = "slowest"
    ) %>%
    mutate(data_name = recode(data_name, !!!DATASET_TITLES_MAIN))

top_headers <- c(" " = 3, "none" = 4, "MCAR" = 4, "MNAR" = 4)
mid_headers <- c(" " = 3, c("Cov." = 2, "S.A." = 2) %>% rep(3))
bot_headers <- c("Dataset", "Difficulty", "Concept Noise", c("Before", "After", "Before", "After") %>% rep(3))

overview_table <- kable_df %>%
        kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = bot_headers,
            format = "latex",
            table.envir = NULL,
            linesep = ""
        ) %>%
        add_header_above(mid_headers, bold = FALSE, escape = FALSE) %>%
        add_header_above(top_headers, bold = FALSE, escape = FALSE) %>%
        row_spec(2:nrow(kable_df)-1, hline_after = TRUE, extra_latex_after = "\n")

In [ ]:
overview_table

In [ ]:
cells_df = table_stats_df %>%
    arrange(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    mutate(metric = recode(metric, !!!c("selective_acc_before" = "sa_before", "selective_acc_after" = "sa_after", "coverage_before" = "cov_before", "coverage_after" = "cov_after"))) %>%
    pivot_wider(
        # names_from = c(metric, c(target_accuracy_label, has_concept_noise)),
        names_from = c(metric),
        values_from = svalue,
        names_vary = "slowest"
    )

In [ ]:
cells_df

In [ ]:
table_df <- cells_df %>%
  # 1. Pivot to a long format
  pivot_longer(
    cols = starts_with(c("cov", "sa")),
    names_to = c("metric", "level", "boolean_value"),
    names_sep = "_",
    values_to = "value"
  ) %>%
  # 2. Re-unite the desired columns based on common metrics and levels
  group_by(data_name, metric, level, concept_missing_mech) %>%
  summarise(
    combined_value = paste(value, collapse = "\\\\")
  ) %>%
  ungroup() %>%
  # 3. Pivot back to the wide format
  pivot_wider(
    names_from = c(metric, level),
    values_from = combined_value,
    names_sep = "_",
    names_glue = "{metric}_{level}"
  ) %>%
  # 4. Final formatting (optional)
  mutate(across(starts_with(c("cov", "sa")), ~ sprintf("\\cell{r}{%s}", .)))

In [ ]:
table_df

In [ ]:
DATASET_TITLES_MAIN <- c(
    "sudoku" = "\\sudokuinfo{}",
    "robot" = "\\robotinfo{}"
)

kable_df <- table_df %>%
    pivot_wider(
        names_from = c("concept_missing_mech"),
        values_from = c("cov_before", "cov_after", "sa_before", "sa_after"),
        names_vary = "slowest"
    ) %>%
    mutate(difficulty = c("\\SudokuDiff{}", "\\RobotDiff{}")) %>%
    relocate(difficulty, .after = data_name) %>%
    mutate(concept_noise = c("\\SudokuConceptNoise{}", "\\RobotConceptNoise{}")) %>%
    relocate(concept_noise, .after = difficulty) %>%
    mutate(data_name = recode(data_name, !!!DATASET_TITLES_MAIN))

top_headers <- c(" " = 3, "\\\\noMissing{}" = 4, "\\\\mcar{}" = 4, "\\\\mnar{}" = 4)
mid_headers <- c(" " = 3, c("Cov." = 2, "S.A." = 2) %>% rep(3))
bot_headers <- c("Dataset", "Difficulty", "Concept Noise", c("Before", "After", "Before", "After") %>% rep(3))

overview_table <- kable_df %>%
        kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = bot_headers,
            format = "latex",
            table.envir = NULL,
            linesep = ""
        ) %>%
        add_header_above(mid_headers, bold = FALSE, escape = FALSE) %>%
        add_header_above(top_headers, bold = FALSE, escape = FALSE) %>%
        row_spec(2:nrow(kable_df)-1, hline_after = TRUE, extra_latex_after = "\n")

In [ ]:
overview_table

In [ ]:
sudoku_results <- "../../../results/big_demo/conceptual_safeguards_sudoku.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.05
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")

df <- read_csv(sudoku_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

top_headers <- c(" ", " ", "Coverage", "Coverage", "Selective Acc.", "Selective Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Before", "After", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 2, "Coverage" = 2, "Selective Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

In [ ]:
robot_cs_results <- "../../../results/big_demo/conceptual_safeguards_robot.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")
difficulty_order = c("easy", "medium", "hard")

df <- read_csv(robot_cs_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)

df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, target_accuracy_label, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, target_accuracy_label, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

top_headers <- c(" ", " ", "", "Coverage", "Coverage", "Selective Acc.", "Selective Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Before", "After", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 3, "Coverage" = 2, "Selective Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

In [ ]:
robot_results <- "../../../results/big_demo/score_intervention_robot.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("intervened", "overall_acc_before", "overall_acc_after", "acc_non_intervened_before")
difficulty_order = c("easy", "medium", "hard")

df <- read_csv(robot_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)
df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, target_accuracy_label, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, target_accuracy_label, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    ) %>%
    select(-acc_non_intervened_before)

top_headers <- c(" ", " ", "", "", "Overall Acc.", "Overall Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Intervened", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 4, "Overall Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

In [ ]:
robot_results <- "../../../results/robot_demo_results.csv"

# CONCEPT_NOISE = 0.15
# CONCEPT_MISSING = 0.3
# SPLIT = "test"

TAU = 0.6
metric_order = c("accuracy", "predictions_intervened_on", "total_concept_edits_made")
model_order = c("dnn", "cbm_no_int", "cbm_with_int_1", "cbm_with_int_3")
DATASET_TITLES <- c(
    "ideal" = "\\robotIdeal{}",
    "subconcept" = "\\robotSubconcept{}"
)

df <- read_csv(robot_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$model <- factor(df$model, levels = model_order)
df <- df %>%
    # mutate(model = ifelse(is.na(budget), model, paste0(model, "_", budget))) %>%
    filter((threshold == TAU) | is.na(threshold)) %>%
    select(data_name, model, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value),
        svalue_int = sprintf("%d", round(value))) %>%
    mutate(svalue = ifelse(metric == "accuracy", svalue_dec,
                        ifelse(metric == "predictions_intervened_on", svalue_int,
                               svalue_int))) %>%
    select(-svalue_pct, -svalue_dec, -svalue_int)

cells_df <- table_stats_df %>%
    arrange(model, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

cells_df[is.na(cells_df)] <- EMPTY_TEX_STRING

table_df <- cells_df %>%
    group_by(data_name, model) %>%
    unite(cell_str, sep = "\\\\", metric_order) %>%
    mutate(cell_str = sprintf("\\cell{r}{%s}\n", cell_str)) %>%
    ungroup() %>%
    arrange(data_name, model)

kable_df <- table_df %>%
    mutate(
        metrics = "\\robotMetrics{}",
        data_name = recode(data_name, !!!DATASET_TITLES)
    ) %>%
    pivot_wider(
        names_from = c(model),
        values_from = cell_str,
        names_sort = FALSE,
        names_glue = "{model}",
    )

kable_df[is.na(kable_df)] <- "\\cell{r}{---\\\\---\\\\---}"

top_headers <- c("Dataset", "Metrics", "DNN", "CBM No Int.", "CBM With Int. (1)", "CBM With Int. (3)")
# mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Intervened", "Before", "After")

table <- kable_df %>%
    kbl(escape = FALSE, toprule = '', align = "l", booktabs = TRUE, linesep = "\\midrule", 
        col.names = top_headers, format="latex")

In [ ]:
robot_results <- "../../../results/robot_demo_results.csv"

# CONCEPT_NOISE = 0.15
# CONCEPT_MISSING = 0.3
# SPLIT = "test"

TAU = 0.2
metric_order = c("accuracy", "total_concept_edits_made")
model_order = c("cbm_no_int", "cbm_with_int_1", "cbm_with_int_3")
missing_order = c("none", "mcar", "mnar")
DATASET_TITLES <- c(
    "ideal" = "\\robotIdeal{}",
    "subconcept" = "\\robotSubconcept{}"
)

df <- read_csv(robot_results, show_col_types = FALSE)
# Calculate relative gain in accuracy compared to baseline DNN
df[df$metric == "accuracy", "value"] <- df[df$metric == "accuracy", "value"] - df[(df$metric == "accuracy") & (df$model == "dnn") & (df$data_name == "ideal"), "value"][[1]]
df <- df %>% filter(model %in% model_order) %>% filter(metric %in% metric_order)
df$metric <- factor(df$metric, levels = metric_order)
df$model <- factor(df$model, levels = model_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df <- df %>%
    # mutate(model = ifelse(is.na(budget), model, paste0(model, "_", budget))) %>%
    filter((threshold == TAU) | is.na(threshold)) %>%
    select(data_name, concept_missing_mech, model, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value),
        svalue_int = sprintf("%d", round(value))) %>%
    mutate(svalue = ifelse(metric == "accuracy", svalue_dec, svalue_int)) %>%
    select(-svalue_pct, -svalue_dec, -svalue_int) %>%
    # prepend and append $
    mutate(svalue = ifelse(metric == "accuracy", paste0("$", svalue, "$"), svalue))

cells_df <- table_stats_df %>%
    arrange(model, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = c("metric"),
        # names_from = concept_missing_mech,
        values_from = svalue,
        values_fill = EMPTY_TEX_STRING
    ) %>%
    pivot_longer(
        cols = -c(data_name, concept_missing_mech, model),
        names_to = "metric",
        values_to = "svalue"
    ) %>%
    pivot_wider(
        names_from = concept_missing_mech,
        values_from = svalue,
        values_fill = EMPTY_TEX_STRING
    ) 

table_df <- cells_df %>%
    group_by(data_name, model) %>%
    unite(cell_str, sep = "\\\\", missing_order) %>%
    mutate(cell_str = sprintf("\\cell{r}{%s}\n", cell_str)) %>%
    ungroup() %>%
    arrange(data_name, model) %>%
    pivot_wider(
        names_from = c("model", "metric"),
        values_from = cell_str,
        names_sort = FALSE,
        names_glue = "{model}+{metric}",
    )

kable_df <- table_df %>%
    mutate(
        missing = "\\robot_missing{}",
        data_name = recode(data_name, !!!DATASET_TITLES)
    ) %>%
    relocate(missing , .after = data_name)

top_headers <- c(" " = 2, "CBM" = 2, "w/ Intervnetion ($k=1$)" = 2, "w/ Intervnetion ($k=3$)" = 2)
mid_headers <- c("Dataset", "Missingness", "Acc. Gain", "Total Work", "Acc. Gain", "Total Work", "Acc. Gain", "Total Work")

table <- kable_df %>%
    kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = mid_headers,
            format = "latex",
            table.envir = NULL,
            linesep = "\\midrule",
            align = c("l", "l", rep("r", 6))
        ) %>%
    add_header_above(header = top_headers, bold=FALSE, escape=FALSE) %>%
    row_spec(0, bold = TRUE)

In [ ]:
table

In [ ]:
robot_results <- "../../../results/robot_demo_results.csv"

# CONCEPT_NOISE = 0.15
# CONCEPT_MISSING = 0.3
# SPLIT = "test"

TAU = 0.2
metric_order = c('accuracy', 'predictions_intervened_on', 'total_concept_confirmations', 'predictions_changed')
model_order = c("dnn", "cbm_no_int", "cbm_with_int_1", "cbm_with_int_3", "cbm_with_int_max")
missing_order = c("none", "mcar", "mnar")
DATASET_TITLES <- c(
    "ideal_none" = "\\robotIdeal{}",
    "ideal_mcar" = "\\robotIdealMCAR{}",
    "ideal_mnar" = "\\robotIdealMNAR{}",
    "subconcept_none" = "\\robotSubconcept{}",
    "subconcept_mcar" = "\\robotSubconceptMCAR{}",
    "subconcept_mnar" = "\\robotSubconceptMNAR{}"
)

df <- read_csv(robot_results, show_col_types = FALSE)
df <- df %>% mutate(model = ifelse((model == "cbm_with_int_7") | (model == "cbm_with_int_12"), "cbm_with_int_max", model))
df <- df %>% filter(model %in% model_order)
df$model <- factor(df$model, levels = model_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df <- df %>%
    # mutate(model = ifelse(is.na(budget), model, paste0(model, "_", budget))) %>%
    filter((threshold == TAU) | is.na(threshold)) %>%
    select(data_name, concept_missing_mech, model, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value),
        svalue_int = sprintf("%d", round(value))) %>%
    mutate(svalue = ifelse(metric == "accuracy", svalue_pct, svalue_int)) %>%
    select(-svalue_pct, -svalue_dec, -svalue_int)

cells_df <- table_stats_df %>%
    arrange(model, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = c("metric"),
        # names_from = concept_missing_mech,
        values_from = svalue,
        values_fill = EMPTY_TEX_STRING
    )

table_df <- cells_df %>%
    group_by(data_name, model, concept_missing_mech) %>%
    unite(cell_str, sep = "\\\\", metric_order) %>%
    mutate(cell_str = sprintf("\\cell{r}{%s}\n", cell_str)) %>%
    ungroup() %>%
    arrange(data_name, model) %>%
    pivot_wider(
        names_from = c("model"),
        values_from = cell_str,
        names_sort = FALSE,
        # names_glue = "{model}",
    ) %>%
    # fill na values
    replace_na(list(dnn="\\cell{r}{---\\\\---\\\\---\\\\---}"))

kable_df <- table_df %>%
    mutate(data_name = paste0(data_name, "_", concept_missing_mech)) %>%
    mutate(data_name = recode(data_name, !!!DATASET_TITLES)) %>%
    relocate(data_name , .before = everything()) %>%
    select(-concept_missing_mech) %>%
    mutate(metrics = "\\robotAllMetrics{}") %>%
    relocate(metrics , .after = data_name)

top_headers <- c(" " = 3, "CBM" = 4)
mid_headers <- c("Dataset", "Metrics", "DNN", "No Int.", "w/ Int. ($k=1$)", "w/ Int. ($k=3$)", "w/ Int. (max)")

table <- kable_df %>%
    kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = mid_headers,
            format = "latex",
            table.envir = NULL,
            linesep = "\\midrule",
            align = c("l", "l", rep("r", 6))
        ) %>%
    add_header_above(header = top_headers, bold=FALSE, escape=FALSE) %>%
    row_spec(0, bold = TRUE)

In [ ]:
table

In [ ]:
kable_df